In [8]:
import importnb
import os
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error

with importnb.Notebook():
    import acquisition_and_profiling as phase_1
    import scaling_and_3DTransformation as phase_3
    import topology as phase_4
    import training_pipline as phase_5

--- INITIATING NETWORK TRAINING (SCALED TARGETS) ---
Epoch 01/15 | Average MSE Loss: 0.0144
Epoch 02/15 | Average MSE Loss: 0.0093
Epoch 03/15 | Average MSE Loss: 0.0079
Epoch 04/15 | Average MSE Loss: 0.0070
Epoch 05/15 | Average MSE Loss: 0.0059
Epoch 06/15 | Average MSE Loss: 0.0048
Epoch 07/15 | Average MSE Loss: 0.0042
Epoch 08/15 | Average MSE Loss: 0.0037
Epoch 09/15 | Average MSE Loss: 0.0033
Epoch 10/15 | Average MSE Loss: 0.0031
Epoch 11/15 | Average MSE Loss: 0.0027
Epoch 12/15 | Average MSE Loss: 0.0026
Epoch 13/15 | Average MSE Loss: 0.0023
Epoch 14/15 | Average MSE Loss: 0.0026
Epoch 15/15 | Average MSE Loss: 0.0021


In [9]:
testing_data_path = phase_1.testing_data_path
rul_data_path = phase_1.rul_data_path
subset = phase_1.subset
columns = phase_1.columns
cols_to_drop = phase_1.cols_to_drop
    
sensor_columns = phase_3.sensor_columns
scaler = phase_3.scaler
WINDOW_SIZE = phase_3.WINDOW_SIZE
    
device = phase_4.device
model = phase_4.model
target_scaler = phase_5.target_scaler # THE DEPENDENCY FIX

# 1. Acquisition & Purging
df_test = pd.read_csv(
    filepath_or_buffer=os.path.join(testing_data_path, f"test_{subset}.txt"),
    sep=r'\s+',
    names=columns,
    index_col=False
)
df_test.set_index(['Engine_ID', 'Cycle'], inplace=True)
df_test.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 2. Strict Scaling (Sensors)
df_test[sensor_columns] = scaler.transform(df_test[sensor_columns])

# 3. 3D Tensor Extraction
X_test_list = []
engine_ids_test = df_test.index.get_level_values('Engine_ID').unique()

for engine_id in engine_ids_test:
    engine_data = df_test.xs(engine_id, level='Engine_ID')[sensor_columns].values
    final_window = engine_data[-WINDOW_SIZE:, :]
    X_test_list.append(final_window)

X_test = np.array(X_test_list)

# 4. Target Engineering (The True Unscaled Answer Key)
true_rul = pd.read_csv(
    filepath_or_buffer=os.path.join(rul_data_path, f"RUL_{subset}.txt"), 
    sep=r'\s+', 
    header=None
)[0].values

Y_test = true_rul

# 5. Inference Execution
model.eval()
tensor_X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

print("\n--- RUNNING INFERENCE ---")
with torch.no_grad():
    # The network predicts a SCALED probability-like number (e.g., 0.35)
    scaled_predictions = model(tensor_X_test).cpu().numpy()
    
    # --- THE ARCHITECTURAL FIX: INVERSE TRANSFORM ---
    # We force the scaler to multiply the 0-1 range back out to the real engine cycle scale
    predictions = target_scaler.inverse_transform(scaled_predictions).flatten()

# 6. The Evaluation Matrix
print("\n--- REGRESSION PERFORMANCE METRICS ---")
mae = mean_absolute_error(Y_test, predictions)
rmse = np.sqrt(mean_squared_error(Y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.2f} cycles")
print(f"Root Mean Square Error (RMSE): {rmse:.2f} cycles")

print("\n--- SAMPLE PREDICTION VERIFICATION (First 5 Engines) ---")
for i in range(5):
    print(f"Engine {i+1} | True RUL: {Y_test[i]} | Predicted RUL: {predictions[i]:.1f} | Error: {abs(Y_test[i] - predictions[i]):.1f}")


--- RUNNING INFERENCE ---

--- REGRESSION PERFORMANCE METRICS ---
Mean Absolute Error (MAE): 24.68 cycles
Root Mean Square Error (RMSE): 37.33 cycles

--- SAMPLE PREDICTION VERIFICATION (First 5 Engines) ---
Engine 1 | True RUL: 112 | Predicted RUL: 171.5 | Error: 59.5
Engine 2 | True RUL: 98 | Predicted RUL: 158.1 | Error: 60.1
Engine 3 | True RUL: 69 | Predicted RUL: 42.1 | Error: 26.9
Engine 4 | True RUL: 82 | Predicted RUL: 65.3 | Error: 16.7
Engine 5 | True RUL: 91 | Predicted RUL: 126.9 | Error: 35.9
